# Transfer Learning

In [22]:
import pathlib
import os
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
# import keras_tuner as kt
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.model_selection import StratifiedKFold, train_test_split


In [23]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

# # Place tensors on the CPU
# with tf.device('/CPU:0'):
#   a = tf.constant([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
#   b = tf.constant([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])

# # Run on the GPU
# c = tf.matmul(a, b)
# print(c)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
  try:
    # Currently, memory growth needs to be the same across GPUs
    for gpu in gpus:
      tf.config.experimental.set_memory_growth(gpu, True)
    logical_gpus = tf.config.list_logical_devices('GPU')
    print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
  except RuntimeError as e:
    # Memory growth must be set before GPUs have been initialized
    print(e)

if gpus:
    try:
        tf.config.experimental.set_virtual_device_configuration(
            gpus[0],
            [tf.config.experimental.VirtualDeviceConfiguration(memory_limit=1024*6)])  # Limit to 6GB
    except RuntimeError as e:
        print(e)

Num GPUs Available:  0


### Import Model

In [24]:
batch_size = 32
img_height = 224
img_width = 224

# configurations that will be used in training
configs = [
    {"learning_rate": 0.001, "optimizer": "adam", "epochs": 100, "save_metrics": True, "fine_tune": False, "fine_tune_epochs": 25, "fine_tune_at": 150},
    # {"learning_rate": 0.001, "optimizer": "adam", "epochs": 50, "save_metrics": True, "fine_tune": False, "fine_tune_epochs": 25, "fine_tune_at": 150},
    # {"learning_rate": 0.001, "optimizer": "adam", "epochs": 50, "save_metrics": True, "fine_tune": True, "fine_tune_epochs": 25, "fine_tune_at": 152},
    # {"learning_rate": 0.001, "optimizer": "adam", "epochs": 50, "save_metrics": True, "fine_tune": True, "fine_tune_epochs": 25, "fine_tune_at": 152},
]

# Define the base path for saving models
save_dir = "../saved_models"
os.makedirs(save_dir, exist_ok=True)


data_root_train = pathlib.Path("../data/carcinoma/archive/train")    # points to the folder containing the images that will be used for training
data_root_val = pathlib.Path("../data/carcinoma/archive/val")    # points to the folder containing the images that will be used for training
data_root_test = pathlib.Path("../data/carcinoma/archive/test")    # points to the folder containing the images that will be used for training


train_ds = tf.keras.utils.image_dataset_from_directory(
  str(data_root_train),
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size
)

val_ds = tf.keras.utils.image_dataset_from_directory(
  str(data_root_val),
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size
)

test_ds = tf.keras.utils.image_dataset_from_directory(
  str(data_root_val),
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size
)

Found 29322 files belonging to 14 classes.
Found 3660 files belonging to 14 classes.
Found 3660 files belonging to 14 classes.


In [25]:
# # Load dataset without splitting
# dataset = tf.keras.utils.image_dataset_from_directory(
#     data_root,                                  # loads images from the data_root directory
#     image_size=(img_height, img_width),         # resizes all images to (224, 224) pixels
#     batch_size=batch_size,                      # set the batch size
#     shuffle=True                                # shufle data when loaded
# )

class_names = np.array(train_ds.class_names)     # get the class names for the data
num_classes = len(class_names)                  # get the number of classes in the dataset

print(num_classes)

# # convert the dataset to a list of (image, label) pairs. This makes it easier to perform cross-validation
# image_paths, labels = [], []
# for image_batch, label_batch in dataset:
#     image_paths.extend(image_batch.numpy())
#     labels.extend(label_batch.numpy())

# image_paths = np.array(image_paths)             # convert to numpy array to facilitate training
# labels = np.array(labels)                       # convert to numpy array to facilitate training

# # Split the dataset into training/validation and test sets
# train_val_images, test_images, train_val_labels, test_labels = train_test_split(
#     image_paths, labels, test_size=0.10, random_state=42, stratify=labels
# )

14


In [ ]:
def callbacks_setup(checkpoint_filepath):
    # EarlyStopping callback configuration
    early_stopping = EarlyStopping(
        monitor='val_loss',        # monitor validation loss
        patience=6,                # number of epochs with no improvement to stop training
        mode = 'min',              # want to minimize what it being monitored 
        restore_best_weights=False # don't restore in EarlyStopping, handled by ModelCheckpoint
    )

    model_checkpoint = ModelCheckpoint(
        filepath=checkpoint_filepath,   # path to save weights
        save_weights_only=True,         # only save weights instead of full model
        monitor='val_accuracy',        # monitor validation loss
        mode='max',                     # want to maximize what is being monitored
        save_best_only=True             # save the best weights
    )

    reduce_lr = ReduceLROnPlateau(
        monitor='val_loss',        # monitor validation loss 
        factor=0.5,                # factor by which the learning rate will be reduced 
        patience=3,                # number of epochs with no improvement to stop training 
        mode = 'min',              # want to minimize what it being monitored 
        min_lr=1e-6                # lower bound on the learning rate 
    )            

    return early_stopping, model_checkpoint, reduce_lr

In [27]:
from sklearn.metrics import precision_score, classification_report, roc_auc_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

# plot and save confusion matrix
def save_confusion_matrix(true_labels, predicted_labels, class_names, save_path):
    cm = confusion_matrix(true_labels, predicted_labels)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(cmap=plt.cm.Blues)
    plt.title("Confusion Matrix")
    plt.savefig(save_path)
    plt.close()

# plot and save loss curves
def save_loss_curve(history, save_path):
    plt.figure(figsize=(10, 6))
    plt.plot(history['loss'], label='Training Loss', color='blue')
    plt.plot(history['val_loss'], label='Validation Loss', color='orange')
    plt.title("Training and Validation Loss Over Epochs")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.savefig(save_path)
    plt.close()

# compute and plot evaluation metrics (accuracy, sensitivity, specificity, F1 score)
def save_evaluation_metrics(true_labels, predicted_labels, history, cm, save_path):
    accuracy = history['val_accuracy'][-1]
    sensitivity = recall_score(true_labels, predicted_labels, average='macro')
    specificity = np.mean(np.diag(cm) / (np.diag(cm) + np.sum(cm, axis=0) - np.diag(cm)))
    f1 = f1_score(true_labels, predicted_labels, average='macro')

    metrics = {
        "Accuracy": accuracy,
        "Sensitivity (Recall)": sensitivity,
        "Specificity": specificity,
        "F1-Score": f1
    }

    plt.figure(figsize=(10, 6))
    plt.bar(metrics.keys(), metrics.values(), color=['darkturquoise', 'sandybrown', 'hotpink', 'limegreen'])
    plt.title("Model Evaluation Metrics")
    plt.ylim([0, 1])
    plt.yticks(np.arange(0, 1.1, 0.1))
    plt.ylabel("Score")
    plt.savefig(save_path)
    plt.close()
    return metrics

# save classification report
def save_classification_report(true_labels, predicted_labels, class_names, save_path):
    class_report = classification_report(true_labels, predicted_labels, target_names=class_names, digits=4)
    with open(save_path, "w") as f:
        f.write(class_report)

# Function to calculate metrics
def calculate_metrics(true_labels, predictions):
    accuracy = np.mean(np.argmax(predictions, axis=1) == true_labels)
    precision = precision_score(true_labels, np.argmax(predictions, axis=1), average='macro')
    recall = recall_score(true_labels, np.argmax(predictions, axis=1), average='macro')
    f1 = f1_score(true_labels, np.argmax(predictions, axis=1), average='macro')
    auc = roc_auc_score(tf.keras.utils.to_categorical(true_labels), predictions, multi_class='ovr')
    return accuracy, precision, recall, f1, auc

# Function to save metrics, loss curve, and confusion matrix for the best model
def save_best_model_visuals(history, model, val_ds, class_names, weights_path, fold):
    # generate predictions for the validation set
    val_predictions = model.predict(val_ds)
    val_predicted_ids = np.argmax(val_predictions, axis=-1)
    true_labels = np.concatenate([y for x, y in val_ds], axis=0)

    # confusion Matrix
    confusion_matrix_path = os.path.join(weights_path, f"confusion_matrix_fold_{fold}.png")
    save_confusion_matrix(true_labels, val_predicted_ids, class_names, confusion_matrix_path)

    # loss curve
    loss_curve_path = os.path.join(weights_path, f"loss_curve_fold_{fold}.png")
    save_loss_curve(history.history, loss_curve_path)

    # evaluation Metrics (Accuracy, Sensitivity, Specificity, F1 Score)
    cm = confusion_matrix(true_labels, val_predicted_ids)
    metrics_bar_chart_path = os.path.join(weights_path, f"evaluation_metrics_fold_{fold}.png")
    save_evaluation_metrics(true_labels, val_predicted_ids, history.history, cm, metrics_bar_chart_path)

    # save classification report as a text file
    classification_report_path = os.path.join(weights_path, f"classification_report_fold_{fold}.txt")
    save_classification_report(true_labels, val_predicted_ids, class_names, classification_report_path)

In [28]:
# Function to create and compile the model
def create_model(num_classes, config, fine_tune=None):
    # if you are not fine tuning the model, instantiate a new model 
    if(fine_tune == False):         
        # instantiate mobilenet (contains 154 layers)
        base_model = tf.keras.applications.MobileNetV2(
            input_shape=(img_height, img_width, 3),     # set the input it will receive
            include_top=False,                          # do not include top layer to perform transfer learning
            weights='imagenet'                          # load weights from imagenet dataset
        )
        base_model.trainable = False                    # Freeze the base model
        
        # add a layer in order to perform classification on our dataset
        model = Sequential([
            base_model,                         # use base_model as the start of your model
            layers.GlobalAveragePooling2D(),    # add a final layer to perform classification
            layers.Dense(num_classes)           # set the number of possible prediction to the num of classes in dataset
        ])
        
    # select optimizer and learning rate based on configuration
    if config["optimizer"] == "adam":
        optimizer = tf.keras.optimizers.Adam(learning_rate=config["learning_rate"])
    elif config["optimizer"] == "sgd":
        optimizer = tf.keras.optimizers.SGD(learning_rate=config["learning_rate"])
    else:
        raise ValueError(f"Unsupported optimizer: {config['optimizer']}")

    # compile the model
    model.compile(
        optimizer=optimizer,
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy']
    )
    
    return model

# fine tune model by unfreezing the layers after the first fine_tune_at layers
def fine_tune_model(base_model, fine_tune_at):
    # Unfreeze the layers starting from fine_tune_at index
    for layer in base_model.layers[:fine_tune_at]:
        layer.trainable = False
    for layer in base_model.layers[fine_tune_at:]:
        layer.trainable = True


In [ ]:
# train_metrics = []      # list to save training metrics
# val_metrics = []        # list to save validation metrics

# fold = 1

# for i, config in enumerate(configs):
#     print(f"Training model {i + 1}/{len(configs)} with config: {config}")

#     # K-fold Cross Validation
#     kfold = StratifiedKFold(n_splits=config['folds'], shuffle=True, random_state=42)
#     best_val_f1score = -float('inf')            # Initialize best F1 score with a very low value

#     # Define the base path for saving models
#     model_subdir = os.path.join(save_dir, f'model{i + 1}')
#     os.makedirs(model_subdir, exist_ok=True)

#     # Define the base path for saving checkpoints for model
#     checkpoint_folder = os.path.join(model_subdir, 'checkpoints')
#     os.makedirs(checkpoint_folder, exist_ok=True)

#     # Define the base path for saving cthe model with the best f1-score
#     best_f1_dir = os.path.join(model_subdir, 'best_f1score_fold')
#     os.makedirs(best_f1_dir, exist_ok=True)
    
#     # Training and validation loop for each fold
#     fold = 1
#     for train_idx, val_idx in kfold.split(train_val_images, train_val_labels):
#         print(f"\nFold {fold}/{config['folds']}...")

#         checkpoint_filepath = os.path.join(checkpoint_folder, f'checkpoint_fold{fold}.weights.h5')

#         # Create subset datasets for training and validation
#         train_images, train_labels = train_val_images[train_idx], train_val_labels[train_idx]
#         val_images, val_labels = train_val_images[val_idx], train_val_labels[val_idx]

#         # Convert NumPy arrays back to TensorFlow datasets
#         train_ds = tf.data.Dataset.from_tensor_slices((train_images, train_labels))
#         val_ds = tf.data.Dataset.from_tensor_slices((val_images, val_labels))

#         # Normalize datasets 
#         normalization_layer = layers.Rescaling(1./255)
#         train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
#         val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))

#         # prefetch data to improve performance by overlapping data preprocessing and model execution and cache the dataset in memory and batch
#         AUTOTUNE = tf.data.AUTOTUNE
#         train_ds = train_ds.batch(batch_size).cache().prefetch(buffer_size=AUTOTUNE)
#         val_ds = val_ds.batch(batch_size).cache().prefetch(buffer_size=AUTOTUNE)

#         # Step 1: Train model with frozen layers
#         print(f"Training with frozen base layers for {config['epochs']} epochs...")

#         # Create and compile model for each fold
#         model = create_model(num_classes, config, fine_tune=False) 

#         # setup callbacks 
#         early_stopping, model_checkpoint = callbacks_setup(checkpoint_filepath)

#         # train the model on the training set until the epochs specified
#         with tf.device('/GPU:0'):
#             history_frozen = model.fit(
#                 train_ds,                                       # dataset used for training
#                 validation_data=val_ds,                         # dataset used for validation
#                 epochs=config['epochs'],                        # epochs used for training
#                 callbacks=[early_stopping, model_checkpoint],   # set early stopping to avoid overfitting
#                 verbose=1
#             )

#         # load the best weights from ModelCheckpoint after training
#         model.load_weights(checkpoint_filepath)

#         if(config["fine_tune"] == True):
#             # Step 2: Unfreeze layers and fine-tune
#             print(f"Unfreezing layers starting from layer {config['fine_tune_at']} for fine-tuning...")
#             fine_tune_model(model.layers[0], config['fine_tune_at'])      # fine tune model

#             # re-compile the model with a lower learning rate for fine-tuning
#             fine_tune_lr = config['learning_rate'] * 0.01

#             model.compile(
#                 optimizer=tf.keras.optimizers.Adam(learning_rate=fine_tune_lr),
#                 loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
#                 metrics=['accuracy']
#             )
                
#             print(f"Fine-tuning for {config['fine_tune_epochs']} epochs...")

#             # setup callbacks again for fine-tuning phase with a unique checkpoint
#             early_stopping, model_checkpoint = callbacks_setup(checkpoint_filepath)
            
#             history_fine_tune = model.fit(
#                 train_ds,                                       # dataset used for training
#                 validation_data=val_ds,                         # dataset used for validation
#                 epochs=config['fine_tune_epochs'],                        # epochs used for training
#                 callbacks=[early_stopping, model_checkpoint],   # set early stopping to avoid overfitting
#                 verbose=1
#             )

#             # load weights after fine-tuning
#             model.load_weights(checkpoint_filepath)

#         # evaluate on validation set after training
#         val_predictions = model.predict(val_ds)
#         avg_val_loss = model.evaluate(val_ds, verbose=0)[0]
#         avg_val_accuracy, avg_val_precision, avg_val_recall, avg_val_f1, avg_val_auc = calculate_metrics(
#             np.concatenate([y for x, y in val_ds]), val_predictions
#         )

#         print(f"\nValidation: \tLoss: {avg_val_loss:.4f}, Accuracy: {avg_val_accuracy:.4f}, Precision: {avg_val_precision:.4f}, Recall: {avg_val_recall:.4f}, F1 Score: {avg_val_f1:.4f}, AUC Score: {avg_val_auc:.4f}")

#         # save the best model based on validation F1 score
#         if avg_val_f1 > best_val_f1score:
#             best_val_f1score = avg_val_f1
#             model.export(best_f1_dir)
#             print(f"Model with best F1 score during Validation with F1 Score of {best_val_f1score:.4f}")

#             if (config['save_metrics'] == True):
#                 #save confusion matrix, loss curve, evaluation metrics for the best model
#                 history = history_frozen
#                 save_best_model_visuals(history, model, val_ds, class_names, model_subdir, fold)

# # save metrics after training
# # np.save(os.path.join(save_dir, 'train_metrics.npy'), train_metrics)
# # np.save(os.path.join(save_dir, 'val_metrics.npy'), val_metrics)

In [ ]:
train_metrics = []      # list to save training metrics
val_metrics = []        # list to save validation metrics

fold = 1

for i, config in enumerate(configs):
    print(f"Training model {i + 1}/{len(configs)} with config: {config}")

    # Define the base path for saving models
    model_subdir = os.path.join(save_dir, f'model{i + 1}')
    os.makedirs(model_subdir, exist_ok=True)

    # Define the base path for saving checkpoints for model
    checkpoint_folder = os.path.join(model_subdir, 'checkpoints')
    os.makedirs(checkpoint_folder, exist_ok=True)

    # Define the base path for saving cthe model with the best f1-score
    best_f1_dir = os.path.join(model_subdir, 'best_f1score_fold')
    os.makedirs(best_f1_dir, exist_ok=True)

    checkpoint_filepath = os.path.join(checkpoint_folder, f'checkpoint_fold{fold}.weights.h5')

    # Normalize datasets 
    normalization_layer = layers.Rescaling(1./255)
    train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
    val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))

    # prefetch data to improve performance by overlapping data preprocessing and model execution and cache the dataset in memory and batch
    AUTOTUNE = tf.data.AUTOTUNE
    train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
    val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

    # Step 1: Train model with frozen layers
    print(f"Training with frozen base layers for {config['epochs']} epochs...")

    # Create and compile model for each fold
    model = create_model(num_classes, config, fine_tune=False) 

    # setup callbacks 
    early_stopping, model_checkpoint, reduce_lr = callbacks_setup(checkpoint_filepath)

    # train the model on the training set until the epochs specified
    # with tf.device('/GPU:0'):
    history_frozen = model.fit(
        train_ds,                                                  # dataset used for training
        validation_data=val_ds,                                    # dataset used for validation
        epochs=config['epochs'],                                   # epochs used for training
        callbacks=[early_stopping, model_checkpoint, reduce_lr],   # set early stopping to avoid overfitting
        verbose=1
    )

    # load the best weights from ModelCheckpoint after training
    model.load_weights(checkpoint_filepath)

    if(config["fine_tune"] == True):
        # Step 2: Unfreeze layers and fine-tune
        print(f"Unfreezing layers starting from layer {config['fine_tune_at']} for fine-tuning...")
        fine_tune_model(model.layers[0], config['fine_tune_at'])      # fine tune model

        # re-compile the model with a lower learning rate for fine-tuning
        fine_tune_lr = config['learning_rate'] * 0.01

        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=fine_tune_lr),
            loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
            metrics=['accuracy']
        )
            
        print(f"Fine-tuning for {config['fine_tune_epochs']} epochs...")

        # setup callbacks again for fine-tuning phase with a unique checkpoint
        early_stopping, model_checkpoint, reduce_lr = callbacks_setup(checkpoint_filepath)
        
        history_fine_tune = model.fit(
            train_ds,                                       # dataset used for training
            validation_data=val_ds,                         # dataset used for validation
            epochs=config['fine_tune_epochs'],                        # epochs used for training
            callbacks=[early_stopping, model_checkpoint, reduce_lr],   # set early stopping to avoid overfitting
            verbose=1
        )

        # load weights after fine-tuning
        model.load_weights(checkpoint_filepath)

    # evaluate on validation set after training
    val_predictions = model.predict(val_ds)
    avg_val_loss = model.evaluate(val_ds, verbose=0)[0]
    avg_val_accuracy, avg_val_precision, avg_val_recall, avg_val_f1, avg_val_auc = calculate_metrics(
        np.concatenate([y for x, y in val_ds]), val_predictions
    )

    print(f"\nValidation: \tLoss: {avg_val_loss:.4f}, Accuracy: {avg_val_accuracy:.4f}, Precision: {avg_val_precision:.4f}, Recall: {avg_val_recall:.4f}, F1 Score: {avg_val_f1:.4f}, AUC Score: {avg_val_auc:.4f}")

    # save the best model based on validation F1 score
    if (config['save_metrics'] == True):
        #save confusion matrix, loss curve, evaluation metrics for the best model
        history = history_frozen
        save_best_model_visuals(history, model, val_ds, class_names, model_subdir, fold)

# save metrics after training
# np.save(os.path.join(save_dir, 'train_metrics.npy'), train_metrics)
# np.save(os.path.join(save_dir, 'val_metrics.npy'), val_metrics)

Training model 1/1 with config: {'learning_rate': 0.001, 'optimizer': 'adam', 'epochs': 100, 'save_metrics': True, 'fine_tune': False, 'fine_tune_epochs': 25, 'fine_tune_at': 150}
Training with frozen base layers for 100 epochs...
Epoch 1/100
917/917 ━━━━━━━━━━━━━━━━━━━━ 221s 234ms/step - accuracy: 0.5860 - loss: 1.2298 - val_accuracy: 0.6893 - val_loss: 0.8579 - learning_rate: 0.0010
Epoch 2/100
917/917 ━━━━━━━━━━━━━━━━━━━━ 232s 253ms/step - accuracy: 0.7132 - loss: 0.8124 - val_accuracy: 0.7049 - val_loss: 0.8019 - learning_rate: 0.0010
Epoch 3/100
917/917 ━━━━━━━━━━━━━━━━━━━━ 271s 295ms/step - accuracy: 0.7360 - loss: 0.7406 - val_accuracy: 0.7148 - val_loss: 0.7784 - learning_rate: 0.0010
Epoch 4/100
917/917 ━━━━━━━━━━━━━━━━━━━━ 260s 283ms/step - accuracy: 0.7497 - loss: 0.6985 - val_accuracy: 0.7205 - val_loss: 0.7669 - learning_rate: 0.0010
Epoch 5/100
917/917 ━━━━━━━━━━━━━━━━━━━━ 230s 251ms/step - accuracy: 0.7608 - loss: 0.6690 - val_accuracy: 0.7221 - val_loss: 0.7614 - learni

KeyboardInterrupt: 

In [1]:
import os
import glob

# Define the base directory containing 'train', 'val', 'test' folders
base_dir = '/path/to/your/dataset'

# Function to count images in subdirectories
def count_images_in_subfolders(base_dir):
    # Loop through the train, val, and test folders
    for split in ['train', 'val', 'test']:
        print(f"Processing {split} folder:")
        split_dir = os.path.join(base_dir, split)

        # Loop through the 14 subfolders within each of train, val, test
        for subfolder in os.listdir(split_dir):
            subfolder_path = os.path.join(split_dir, subfolder)

            # Check if it's a directory (skip if not)
            if os.path.isdir(subfolder_path):
                # Use glob to find all image files in this subfolder (you can adjust file types like .jpg, .png)
                image_files = glob.glob(os.path.join(subfolder_path, '*.*'))
                image_files = [f for f in image_files if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.gif'))]

                # Print the number of images in the subfolder
                print(f"  {subfolder}: {len(image_files)} images")

# Call the function with the path to your base directory
count_images_in_subfolders(base_dir)


Processing train folder:
  Actinic keratoses: 693 images
  Basal cell carcinoma: 2658 images
  Benign keratosis-like lesions: 2099 images
  Chickenpox: 900 images
  Cowpox: 792 images
  Dermatofibroma: 191 images
  Healthy: 1368 images
  HFMD: 1932 images
  Measles: 660 images
  Melanocytic nevi: 10300 images
  Melanoma: 3617 images
  Monkeypox: 3408 images
  Squamous cell carcinoma: 502 images
  Vascular lesions: 202 images
Processing val folder:
  Actinic keratoses: 86 images
  Basal cell carcinoma: 332 images
  Benign keratosis-like lesions: 262 images
  Chickenpox: 112 images
  Cowpox: 99 images
  Dermatofibroma: 23 images
  Healthy: 171 images
  HFMD: 241 images
  Measles: 82 images
  Melanocytic nevi: 1287 images
  Melanoma: 452 images
  Monkeypox: 426 images
  Squamous cell carcinoma: 62 images
  Vascular lesions: 25 images
Processing test folder:
  Actinic keratoses: 88 images
  Basal cell carcinoma: 333 images
  Benign keratosis-like lesions: 263 images
  Chickenpox: 113 image